# Create Fysiografen Awards (Kungliga Fysiografiska Sällskapet i Lund)

Creates awards of the Royal Physiographic Society in Lund from the only
grant-level lists the society publishes on fysiografen.se:

1. **Yngre forskares resor** (travel grants for young researchers), the
   decision lists on "Anslagstilldelning"
   (https://www.fysiografen.se/sv/stipendierochanslag/Anslagstilldelning/):
   10 rounds 2024-02 to 2026-04 (9 PDFs, 1 xlsx). Recipient, granted SEK
   amount, and in two rounds the trip type or department. Parsed sums equal
   the "Summa/Totalt" printed on 8 of the 9 PDF lists (the 2025-09-16 list
   prints a Summa of 304,357 that does not match its own 36 rows, SEK 679,729).
   The 2024-04-20 round is an xlsx of names without amounts (amount NULL).
2. **Forskningsanslaget Horisont**, previous recipients 2023-2025 (6, name and
   institution, no amount).

**446 travel grants + 6 Horisont grants = 452 awards, 2023-2026, SEK 6.76M.**

**Not covered (not published):** the society's main research grants
("Forskningsanslag" from ~20 named funds). The "Anslagsförteckning" page the
tracker pointed at lists the funds and their rules, not recipients. Citing
works carry 5-digit application numbers for those grants (177 of 226 distinct
stubs for F4320323415, e.g. 40730) that appear nowhere on the site, so none
of the existing stubs collapse onto this ingest.

**Scope:** the travel grants are research support for doctoral students and
young researchers (conferences, field work, visits to other institutions):
kept per the keep-all-pathways rule (`funding_type travel`, flagged).

**Mapping:** `display_name` is composed '<programme>: <recipient> (<date or
year>)' (no titles published). `start_date` = decision date (the date printed
in the list's own heading wins over the link text: two 2025 links carry each
other's dates). Names are 'Family, Given' except the 2026-04 list ('Given
Family'). `funder_award_id`: synthetic `KFS-YFR-<decision date>-<name slug>` /
`KFS-HORISONT-<year>-<name slug>`.

**Prerequisites:** run `scripts/local/fysiografen_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/fysiografen/fysiografen_projects.parquet` (§1.4 shrink guard).

**Funder (Path A, F4320* Crossref-registered):** funder_id `4320323415`,
Kungliga Fysiografiska Sällskapet i Lund (ror 05rnj0j91, doi
10.13039/501100005753), read from `openalex.funders.funders` by id. No twin row.

**Priority:** `583` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.fysiografen_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/fysiografen/fysiografen_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_rows, COUNT(DISTINCT funder_award_id) as distinct_ids
FROM openalex.awards.fysiografen_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.fysiografen_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.fysiografen_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320323415 must resolve to exactly 1 row in `openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320323415;

## Step 2: Create Fysiografen Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.fysiografen_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320323415  -- Kungliga Fysiografiska Sällskapet i Lund
),
src AS (
    SELECT
        TRIM(funder_award_id) AS funder_award_id,
        NULLIF(TRIM(programme), '') AS programme,
        NULLIF(TRIM(recipient_raw), '') AS recipient,
        TRY_TO_DATE(decision_date, 'yyyy-MM-dd') AS decision_date,
        TRY_CAST(year AS INT) AS award_year,
        TRY_CAST(amount AS DOUBLE) AS amount,
        NULLIF(TRIM(funding_type), '') AS funding_type,
        NULLIF(TRIM(lead_given_name), '') AS given_name,
        NULLIF(TRIM(lead_family_name), '') AS family_name,
        NULLIF(TRIM(institution), '') AS institution,
        source_url
    FROM openalex.awards.fysiografen_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
    CONCAT(g.programme, ': ', g.recipient, ' (',
           COALESCE(CAST(g.decision_date AS STRING), CAST(g.award_year AS STRING)), ')') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    g.funder_award_id,
    g.amount,
    CASE WHEN g.amount IS NOT NULL THEN 'SEK' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    COALESCE(g.funding_type, 'research') as funding_type,
    g.programme as funder_scheme,
    'fysiografen_anslagstilldelning' as provenance,
    g.decision_date as start_date,
    CAST(NULL AS DATE) as end_date,
    COALESCE(YEAR(g.decision_date), g.award_year) as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN g.family_name IS NOT NULL THEN named_struct(
        'given_name', g.given_name,
        'family_name', g.family_name,
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', CASE WHEN g.institution IS NOT NULL THEN named_struct(
            'name', g.institution,
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        ) END
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.source_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM src g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- §2.1.1 shape check: synthetic KFS- keys only; expect 0 rows.
SELECT funder_award_id FROM openalex.awards.fysiografen_awards
WHERE NOT funder_award_id RLIKE '^KFS-(YFR-[0-9]{4}-[0-9]{2}-[0-9]{2}|HORISONT-[0-9]{4})-[a-z0-9-]+$'
LIMIT 20;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fysiografen_anslagstilldelning' AND priority = 583;

-- Priority 583: direct-from-funder ingest of the society's published grant lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    583 as priority
FROM openalex.awards.fysiografen_awards
-- Kyle 2026-10-01: travel and conference/symposium grants are not research funding (Conquer Cancer/ASH rule): not shipped.
WHERE funding_type NOT IN ('travel', 'conference');

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.fysiografen_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.fysiografen_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, start_date, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_sek
FROM openalex.awards.fysiografen_awards
GROUP BY funder_scheme, start_date ORDER BY start_date;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.fysiografen_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage (expect 100% title/PI, ~84% amount: 2024-04-20 round and Horisont publish none).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    collect_set(currency) as currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.fysiografen_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fysiografen_anslagstilldelning'
GROUP BY provenance, priority;